In [610]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

# directory that contains agent_tools.py (must run before importing agent_tools)
_pkg = Path.cwd()
# print(_pkg)
if (_pkg / "agent_tools.py").is_file():
    pkg = _pkg
else:
    pkg = _pkg.parent  # typical: cwd is .../notebook, module is one level up

sys.path.insert(0, str(pkg.resolve()))

# Drop stale cache so edits to agent_tools.py are picked up without restarting the kernel
sys.modules.pop("agent_tools", None)
import agent_tools as at
from agent_tools import AgentMemory


# Input Data
Loading dataframe from a parquet or csv data located in path

In [680]:
path_data = '..\data\heloc_dataset_v1.parquet'

<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\arifn\AppData\Local\Temp\ipykernel_6588\1931592331.py:1: SyntaxWarning: invalid escape sequence '\d'
  path_data = '..\data\heloc_dataset_v1.parquet'


In [681]:
try:
    df = pd.read_parquet(path_data)
except:
    try:
        df = pd.read_csv(path_data)
    except:
        raise ValueError(f"Failed to load data from {path_data}")

# Metadata
Setting metadata for column aliases, target column, feature column, time related column, data type (for data split), etc. The purpose of this aliases is so that we don't need to manually change the column name one-by-obne if we want to use different column for label, features, etc.  

col_time: datetime of event. e.g, date of application, click date, etc.  
col_target: column name of groundtruth label.  
cols_feat: column name of features used for training.  
col_day: column name of day-truncated col_time.  
col_month: column name of month-truncated col_time.  
col_type: column name of data type, whether the data is for train, test, valid, or hold-out sample (hoot, oot).  
col_score: column name of predicted probability score.
cols_feat_num: column name of numerical features.  
cols_feat_cat: column name of categorical features.  
cols_feat_time: column name of datetime features.


In [683]:
col_time = 'date'
col_target = 'label'
cols_feat = ['ExternalRiskEstimate', 'MSinceOldestTradeOpen',
       'MSinceMostRecentTradeOpen', 'AverageMInFile', 'NumSatisfactoryTrades',
       'NumTrades60Ever2DerogPubRec', 'NumTrades90Ever2DerogPubRec',
       'PercentTradesNeverDelq', 'MSinceMostRecentDelq',
       'MaxDelq2PublicRecLast12M', 'MaxDelqEver', 'NumTotalTrades',
       'NumTradesOpeninLast12M', 'PercentInstallTrades',
       'MSinceMostRecentInqexcl7days', 'NumInqLast6M', 'NumInqLast6Mexcl7days',
       'NetFractionRevolvingBurden', 'NetFractionInstallBurden',
       'NumRevolvingTradesWBalance', 'NumInstallTradesWBalance',
       'NumBank2NatlTradesWHighUtilization', 'PercentTradesWBalance']
col_day = 'day'
col_month = 'month'
col_type = 'type'
col_score = 'score'

In [ ]:
df[col_day] = at.get_day_from_date(df, col_time)
df[col_month] = at.get_month_from_date(df, col_time)
cols_feat_num, cols_feat_cat, cols_feat_time = at.get_feature_dtype(df, cols_feat)


# EDA
Explarotary Data Analysis: Understanding the data quality, feature distribution, and labels before start modelling

## Missing rate of features

In [685]:
df_nan_rate = at.get_nan_rate(df, cols_feat)
display(df_nan_rate)

,feature_name,missing_rate
0,ExternalRiskEstimate,0.0
1,MSinceOldestTradeOpen,0.0
2,MSinceMostRecentTradeOpen,0.0
3,AverageMInFile,0.0
4,NumSatisfactoryTrades,0.0
5,NumTrades60Ever2DerogPubRec,0.0
6,NumTrades90Ever2DerogPubRec,0.0
7,PercentTradesNeverDelq,0.0
8,MSinceMostRecentDelq,0.0
9,MaxDelq2PublicRecLast12M,0.0


## Missing rate of features over period of time.

In [686]:
df_nan_rate_timely = at.get_nan_rate_timely(df, cols_feat, col_month)
display(df_nan_rate_timely)

missing_rate                              \
time_period                              201501 201502 201503 201504 201505   
feature_name                                                                  
AverageMInFile                              0.0    0.0    0.0    0.0    0.0   
ExternalRiskEstimate                        0.0    0.0    0.0    0.0    0.0   
MSinceMostRecentDelq                        0.0    0.0    0.0    0.0    0.0   
MSinceMostRecentInqexcl7days                0.0    0.0    0.0    0.0    0.0   
MSinceMostRecentTradeOpen                   0.0    0.0    0.0    0.0    0.0   
MSinceOldestTradeOpen                       0.0    0.0    0.0    0.0    0.0   
MaxDelq2PublicRecLast12M                    0.0    0.0    0.0    0.0    0.0   
MaxDelqEver                                 0.0    0.0    0.0    0.0    0.0   
NetFractionInstallBurden                    0.0    0.0    0.0    0.0    0.0   
NetFractionRevolvingBurden                  0.0    0.0    0.0    0.0    0.0   
NumBank2NatlTradesWHighUtilization          0.0    0.0    0.0    0.0    0.0   
NumInqLast6M                                0.0    0.0    0.0    0.0    0.0   
NumInqLast6Mexcl7days                       0.0    0.0    0.0    0.0    0.0   
NumInstallTradesWBalance                    0.0    0.0    0.0    0.0    0.0   
NumRevolvingTradesWBalance                  0.0    0.0    0.0    0.0    0.0   
NumSatisfactoryTrades                       0.0    0.0    0.0    0.0    0.0   
NumTotalTrades                              0.0    0.0    0.0    0.0    0.0   
NumTrades60Ever2DerogPubRec                 0.0    0.0    0.0    0.0    0.0   
NumTrades90Ever2DerogPubRec                 0.0    0.0    0.0    0.0    0.0   
NumTradesOpeninLast12M                      0.0    0.0    0.0    0.0    0.0   
PercentInstallTrades                        0.0    0.0    0.0    0.0    0.0   
PercentTradesNeverDelq                      0.0    0.0    0.0    0.0    0.0   
PercentTradesWBalance                       0.0    0.0    0.0    0.0    0.0   

                                                                              \
time_period                        201506 201507 201508 201509 201510 201511   
feature_name                                                                   
AverageMInFile                        0.0    0.0    0.0    0.0    0.0    0.0   
ExternalRiskEstimate                  0.0    0.0    0.0    0.0    0.0    0.0   
MSinceMostRecentDelq                  0.0    0.0    0.0    0.0    0.0    0.0   
MSinceMostRecentInqexcl7days          0.0    0.0    0.0    0.0    0.0    0.0   
MSinceMostRecentTradeOpen             0.0    0.0    0.0    0.0    0.0    0.0   
MSinceOldestTradeOpen                 0.0    0.0    0.0    0.0    0.0    0.0   
MaxDelq2PublicRecLast12M              0.0    0.0    0.0    0.0    0.0    0.0   
MaxDelqEver                           0.0    0.0    0.0    0.0    0.0    0.0   
NetFractionInstallBurden              0.0    0.0    0.0    0.0    0.0    0.0   
NetFractionRevolvingBurden            0.0    0.0    0.0    0.0    0.0    0.0   
NumBank2NatlTradesWHighUtilization    0.0    0.0    0.0    0.0    0.0    0.0   
NumInqLast6M                          0.0    0.0    0.0    0.0    0.0    0.0   
NumInqLast6Mexcl7days                 0.0    0.0    0.0    0.0    0.0    0.0   
NumInstallTradesWBalance              0.0    0.0    0.0    0.0    0.0    0.0   
NumRevolvingTradesWBalance            0.0    0.0    0.0    0.0    0.0    0.0   
NumSatisfactoryTrades                 0.0    0.0    0.0    0.0    0.0    0.0   
NumTotalTrades                        0.0    0.0    0.0    0.0    0.0    0.0   
NumTrades60Ever2DerogPubRec           0.0    0.0    0.0    0.0    0.0    0.0   
NumTrades90Ever2DerogPubRec           0.0    0.0    0.0    0.0    0.0    0.0   
NumTradesOpeninLast12M                0.0    0.0    0.0    0.0    0.0    0.0   
PercentInstallTrades                  0.0    0.0    0.0    0.0    0.0    0.0   
PercentTradesNeverDelq                0.0    0.0    0.0    0

### Target (groundtruth) rate over period of time

In [689]:
df_target_rate_timely = at.get_timely_binary_target_rate(df, col_month, col_target)

In [690]:
display(df_target_rate_timely)

,mean,count
month,,
201501,0.480534,899
201502,0.396552,812
201503,0.353726,899
201504,0.480460,870
201505,0.583982,899
201506,0.489655,870
201507,0.474972,899
201508,0.527374,895
201509,0.386905,840


# Split Data

oot: Out of time sample, to check the performance of the model after development data.  
hoot: Historical out of time sample, to check the performance of the model before the development data.
train: data sample for training model
valid: data sample for hyperparameter tuning
test: data sample for on-paper evaluation.

In [ ]:
oot_th = 201510
hoot_th = 201503

In [ ]:
# Use the same integer scale for `col_period` and for oot_th / hoot_th (here: `month`).
df[col_type] = at.split_data(df, col_target, col_month, oot_th, hoot_th)

In [ ]:
df[col_type].value_counts()

type
train    2636
hoot     2610
oot      2576
test     1319
valid    1318
Name: count, dtype: int64

Checking the consistency of the label distribution in each sample type.

In [ ]:
df_target_rate_per_sample = at.get_target_rate_sample(df, col_type, col_target)
display(df_target_rate_per_sample)


,count,mean
type,,
hoot,2610,0.410728
oot,2576,0.518245
test,1319,0.492039
train,2636,0.491654
valid,1318,0.491654


# Feature transformation
Transfor the feature into Weight of Evidence to handle non-monotinicity and inbalance label.

In [ ]:
dict_bin, bp, df_binning_stats, binning_feature_issues = at.get_optimal_bin(
    df.loc[df[col_type] == "train"], cols_feat + ['X'], col_target, cols_feat_cat=cols_feat_cat
)

In [ ]:
display(df_binning_stats.sort_values(by='gini_power', ascending=False))

,name,dtype,status,selected,n_bins,iv,js,gini,quality_score,gini_power
11,NumTotalTrades,numerical,OPTIMAL,True,6,0.024596,0.003058,0.079225,0.002017,0.920775
2,MSinceMostRecentTradeOpen,numerical,OPTIMAL,True,6,0.035198,0.004382,0.100193,0.018778,0.899807
20,NumInstallTradesWBalance,numerical,OPTIMAL,True,6,0.042386,0.005271,0.108321,0.00329,0.891679
12,NumTradesOpeninLast12M,numerical,OPTIMAL,True,6,0.048713,0.006021,0.111746,0.010252,0.888254
4,NumSatisfactoryTrades,numerical,OPTIMAL,True,6,0.074214,0.009178,0.136911,0.064841,0.863089
6,NumTrades90Ever2DerogPubRec,numerical,OPTIMAL,True,3,0.126368,0.01552,0.143345,0.148225,0.856655
18,NetFractionInstallBurden,numerical,OPTIMAL,True,6,0.070341,0.008731,0.14392,0.031213,0.85608
19,NumRevolvingTradesWBalance,numerical,OPTIMAL,True,6,0.086865,0.010744,0.158586,0.160678,0.841414
16,NumInqLast6Mexcl7days,numerical,OPTIMAL,True,5,0.113001,0.013863,0.168369,0.106171,0.831631
15,NumInqLast6M,numerical,OPTIMAL,True,5,0.129148,0.015784,0.180347,0.059235,0.819653


In [ ]:
X_woe, cols_feat_woe, features_issue_woe = at.get_woe_from_bp(df, cols_feat, bp)

In [ ]:
df_binning_tables, features_issue_binning_tables = at.get_binning_tables_from_bp(bp, cols_feat + ['x'])

In [ ]:
df_binning_tables

,Feature Name,Bin,Count,Count (%),Non-event,Event,Event rate,WoE,IV,JS
0,ExternalRiskEstimate,"(-inf, 64.50)",676,0.256449,525,151,0.223373,1.212731,0.333839,0.039347
1,ExternalRiskEstimate,"[64.50, 68.50)",364,0.138088,246,118,0.324176,0.70126,0.064889,0.007949
2,ExternalRiskEstimate,"[68.50, 70.50)",170,0.064492,102,68,0.400000,0.372078,0.008800,0.001094
3,ExternalRiskEstimate,"[70.50, 74.50)",341,0.129363,175,166,0.486804,0.019411,0.000049,0.000006
4,ExternalRiskEstimate,"[74.50, 80.50)",450,0.170713,158,292,0.648889,-0.647546,0.069545,0.008544
...,...,...,...,...,...,...,...,...,...,...
190,PercentTradesWBalance,"[84.50, 89.50)",163,0.061836,108,55,0.337423,0.641411,0.024475,0.003008
191,PercentTradesWBalance,"[89.50, inf)",393,0.149090,298,95,0.241730,1.10983,0.165460,0.019682
192,PercentTradesWBalance,Special,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000
193,PercentTradesWBalance,Missing,0,0.000000,0,0,0.000000,0.0,0.000000,0.000000


In [ ]:
df[cols_feat_woe] = bp.transform(df[cols_feat], metric="woe")

In [ ]:
df[cols_feat_woe]

,ExternalRiskEstimate_woe,MSinceOldestTradeOpen_woe,MSinceMostRecentTradeOpen_woe,AverageMInFile_woe,NumSatisfactoryTrades_woe,NumTrades60Ever2DerogPubRec_woe,NumTrades90Ever2DerogPubRec_woe,PercentTradesNeverDelq_woe,MSinceMostRecentDelq_woe,MaxDelq2PublicRecLast12M_woe,...,PercentInstallTrades_woe,MSinceMostRecentInqexcl7days_woe,NumInqLast6M_woe,NumInqLast6Mexcl7days_woe,NetFractionRevolvingBurden_woe,NetFractionInstallBurden_woe,NumRevolvingTradesWBalance_woe,NumInstallTradesWBalance_woe,NumBank2NatlTradesWHighUtilization_woe,PercentTradesWBalance_woe
0,1.212731,0.026363,0.068578,-0.308196,-0.176618,0.982534,-0.179943,0.980232,1.145268,0.959119,...,0.077839,0.431781,-0.336157,-0.293976,0.204033,-0.212588,0.638483,-0.083471,0.254295,0.133993
1,1.212731,1.022666,-0.132759,0.793657,0.675509,0.982534,0.818584,-0.459261,-0.478819,0.283709,...,0.903738,0.431781,-0.336157,-0.293976,-0.723471,-0.212588,-0.017700,-0.415595,-0.423027,-0.834669
2,0.701260,1.022666,0.068578,0.793657,0.432487,-0.266698,-0.179943,-0.459261,-0.478819,-0.494237,...,0.077839,0.431781,0.425731,0.477439,0.429050,0.193523,-0.147205,-0.043969,0.254295,0.641411
3,0.701260,0.026363,0.107014,-0.051872,-0.176618,0.515046,0.634442,0.202690,-0.371713,0.024784,...,0.586148,0.431781,1.011981,0.477439,1.235124,0.226612,0.173665,0.324450,0.760856,1.109830
4,-1.352153,-0.669376,-0.357437,-0.639523,-0.012768,-0.266698,-0.179943,-0.459261,-0.478819,-0.494237,...,-0.136823,0.431781,0.038934,0.011598,0.429050,0.226612,-0.147205,-0.083471,-0.423027,0.361089
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10454,0.019411,0.526229,0.068578,0.462762,-0.176618,-0.266698,-0.179943,0.202690,-0.371713,0.024784,...,-0.341259,-0.456448,-0.336157,-0.293976,-0.362160,-0.212588,0.173665,-0.043969,-0.423027,1.109830
10455,0.701260,0.026363,-0.357437,-0.051872,0.272780,-0.266698,-0.179943,0.202690,0.089215,0.024784,...,0.047152,-0.044198,0.038934,0.011598,1.495328,-0.048350,-0.390585,-0.043969,0.254295,0.361089
10456,0.019411,0.526229,0.068578,0.462762,-0.012768,0.515046,0.634442,-0.459261,-0.478819,0.024784,...,-0.136823,-0.456448,0.425731,0.477439,-0.723471,-0.212588,0.173665,-0.415595,-0.423027,-0.347168
10457,0.019411,-0.217391,-0.132759,-0.639523,-0.365827,0.802181,0.818584,-0.459261,0.089215,0.024784,...,-0.341259,-0.456448,-0.336157,-0.293976,-0.362160,-0.212588,-0.147205,-0.083471,-0.423027,-0.834669


# Feature Selection

In [ ]:
selected_feats, feats_stats = at.select_features_auc_max_corr(df.loc[df[col_type]=='train'], cols_feat_woe, col_target)

In [634]:
feats_stats

,feature_name,max_correlation,auc_roc,standardized_auc_roc
0,ExternalRiskEstimate_woe,0.493185,0.252408,0.747592
1,PercentTradesWBalance_woe,0.409334,0.334096,0.665904
2,AverageMInFile_woe,0.335007,0.351952,0.648048
3,MSinceMostRecentInqexcl7days_woe,0.349456,0.358304,0.641696
4,NumBank2NatlTradesWHighUtilization_woe,0.493185,0.376375,0.623625
5,NumTrades60Ever2DerogPubRec_woe,0.446790,0.402545,0.597455
6,PercentInstallTrades_woe,0.409334,0.403380,0.596620
7,NumInqLast6M_woe,0.349456,0.409826,0.590174
8,NumRevolvingTradesWBalance_woe,0.469219,0.420707,0.579293
9,NetFractionInstallBurden_woe,0.375085,0.428040,0.571960


In [ ]:
selected_feats, feats_stats = at.select_features_iv_max_corr(df.loc[df[col_type]=='train'], cols_feat_woe, col_target)

In [636]:
feats_stats

,feature_name,max_correlation,iv,auc_roc,standardized_auc_roc
0,ExternalRiskEstimate_woe,0.493185,0.809552,0.252408,0.747592
1,PercentTradesWBalance_woe,0.409334,0.342842,0.334096,0.665904
2,MSinceMostRecentInqexcl7days_woe,0.349456,0.302198,0.358304,0.641696
3,AverageMInFile_woe,0.335007,0.286387,0.351952,0.648048
4,NumBank2NatlTradesWHighUtilization_woe,0.493185,0.150515,0.376375,0.623625
5,PercentInstallTrades_woe,0.409334,0.120870,0.403380,0.596620
6,NumTrades60Ever2DerogPubRec_woe,0.446790,0.101090,0.402545,0.597455
7,NumInqLast6M_woe,0.349456,0.093200,0.409826,0.590174
8,NumRevolvingTradesWBalance_woe,0.469219,0.077814,0.420707,0.579293
9,NumSatisfactoryTrades_woe,0.399495,0.069875,0.431545,0.568455


In [ ]:
selected_feats, df_stats = at.select_features_aic_forward(df.loc[df[col_type]=='train'], cols_feat_woe, col_target)

In [638]:
df_stats

,step_number,added_feature_name,max_correlation,total_aic,delta_aic,roc_auc,gini
0,1,ExternalRiskEstimate_woe,0.000000,3119.804403,-535.733053,0.747592,0.495183
1,2,MSinceMostRecentInqexcl7days_woe,0.202979,2993.054534,-126.749869,0.778171,0.556342
2,3,MSinceOldestTradeOpen_woe,0.223233,2946.605089,-46.449445,0.787343,0.574687
3,4,PercentInstallTrades_woe,0.181983,2918.171189,-28.433901,0.792689,0.585378
4,5,NumSatisfactoryTrades_woe,0.175380,2904.127391,-14.043797,0.795495,0.590991
5,6,NumRevolvingTradesWBalance_woe,0.315671,2888.760944,-15.366447,0.798708,0.597416
6,7,PercentTradesWBalance_woe,0.409334,2887.536067,-1.224877,0.799233,0.598465
7,8,NumInstallTradesWBalance_woe,0.399495,2887.187151,-0.348916,0.799664,0.599329
8,9,NetFractionInstallBurden_woe,0.375085,2887.048797,-0.138354,0.800065,0.600130


In [ ]:
selected_feats, df_stats = at.select_features_aic_backward(df.loc[df[col_type]=='train'], cols_feat_woe, col_target)

In [640]:
df_stats

,step_number,eliminated_feature_name,max_correlation,total_aic,delta_aic,roc_auc,gini
0,1,NumInqLast6M_woe,0.493185,2888.037069,-1.104850,0.801049,0.602097
1,2,NetFractionInstallBurden_woe,0.493185,2887.615001,-0.422068,0.800576,0.601153
2,3,NumTrades60Ever2DerogPubRec_woe,0.493185,2887.346543,-0.268459,0.800578,0.601156
3,4,NumBank2NatlTradesWHighUtilization_woe,0.409334,2887.337191,-0.009352,0.799995,0.599990
4,5,MSinceMostRecentTradeOpen_woe,0.409334,2887.187151,-0.150040,0.799664,0.599329


In [641]:
selected_feats

['ExternalRiskEstimate_woe',
 'MSinceOldestTradeOpen_woe',
 'NumSatisfactoryTrades_woe',
 'PercentInstallTrades_woe',
 'MSinceMostRecentInqexcl7days_woe',
 'NumRevolvingTradesWBalance_woe',
 'NumInstallTradesWBalance_woe',
 'PercentTradesWBalance_woe']

In [ ]:
selected_feats, df_stats = at.select_features_bic_backward(df.loc[df[col_type]=='train'], cols_feat_woe, col_target)

In [643]:
df_stats

,step_number,eliminated_feature_name,max_correlation,total_bic,delta_bic,roc_auc,gini
0,1,NumInqLast6M_woe,0.493185,2964.438302,-6.981868,0.801049,0.602097
1,2,NetFractionInstallBurden_woe,0.493185,2958.139216,-6.299086,0.800576,0.601153
2,3,NumTrades60Ever2DerogPubRec_woe,0.493185,2951.993740,-6.145476,0.800578,0.601156
3,4,NumBank2NatlTradesWHighUtilization_woe,0.409334,2946.107370,-5.886370,0.799995,0.599990
4,5,MSinceMostRecentTradeOpen_woe,0.409334,2940.080312,-6.027058,0.799664,0.599329
5,6,NumInstallTradesWBalance_woe,0.409334,2934.552210,-5.528102,0.799233,0.598465
6,7,PercentTradesWBalance_woe,0.315671,2929.900069,-4.652141,0.798708,0.597416


In [644]:
selected_feats

['ExternalRiskEstimate_woe',
 'MSinceOldestTradeOpen_woe',
 'NumSatisfactoryTrades_woe',
 'PercentInstallTrades_woe',
 'MSinceMostRecentInqexcl7days_woe',
 'NumRevolvingTradesWBalance_woe']

In [ ]:
selected_feats, df_stats = at.select_features_bic_forward(df.loc[df[col_type]=='train'], cols_feat_woe, col_target)

In [646]:
df_stats

,step_number,added_feature_name,max_correlation,total_bic,delta_bic,roc_auc,gini
0,1,ExternalRiskEstimate_woe,0.000000,3131.558439,-529.856035,0.747592,0.495183
1,2,MSinceMostRecentInqexcl7days_woe,0.202979,3010.685588,-120.872851,0.778171,0.556342
2,3,MSinceOldestTradeOpen_woe,0.223233,2970.113161,-40.572427,0.787343,0.574687
3,4,PercentInstallTrades_woe,0.181983,2947.556278,-22.556883,0.792689,0.585378
4,5,NumSatisfactoryTrades_woe,0.175380,2939.389498,-8.166780,0.795495,0.590991
5,6,NumRevolvingTradesWBalance_woe,0.315671,2929.900069,-9.489429,0.798708,0.597416


In [ ]:
selected_feats, df_stats = at.select_features_auc_forward(df.loc[df[col_type]=='train'], cols_feat_woe, col_target)

In [648]:
df_stats

,step_number,feature_name,max_correlation,delta_auc,roc_auc,gini
0,1,ExternalRiskEstimate_woe,0.000000,0.247592,0.747592,0.495183
1,2,MSinceMostRecentInqexcl7days_woe,0.202979,0.030579,0.778171,0.556342
2,3,MSinceOldestTradeOpen_woe,0.223233,0.009173,0.787343,0.574687
3,4,PercentInstallTrades_woe,0.181983,0.005345,0.792689,0.585378


# Model training

In [ ]:
dict_statistics, best_model, importance = at.train_logreg_l1_tune_cv(df, cols_feat_woe, col_target, col_type)

In [ ]:
dict_statistics, best_model2, importance = at.train_logreg_l1_tune_cv(df, selected_feats, col_target, col_type)

In [651]:
dict_statistics

{'best_l1_param': 0.1,
 'train': {'best_auc': 0.7924958540630183,
  'best_gini': 0.5849917081260365,
  'best_l1_param': 0.1,
  'average_auc': 0.7910516747270478,
  'average_gini': 0.5821033494540957},
 'valid': {'best_auc': 0.786375990418279,
  'best_gini': 0.572751980836558,
  'best_l1_param': 0.1,
  'average_auc': 0.6987536852773172,
  'average_gini': 0.3975073705546343},
 'test': {'best_auc': 0.7935469033875306,
  'best_gini': 0.5870938067750613,
  'best_l1_param': 0.1,
  'average_auc': 0.7935469033875306,
  'average_gini': 0.5870938067750613},
 'by_hyperparameter': [{'C': 1e-05,
   'train_cv_fold_aucs': [0.5, 0.5, 0.5, 0.5, 0.5],
   'train_cv_fold_ginis': [0.0, 0.0, 0.0, 0.0, 0.0],
   'train_cv_mean_auc': 0.5,
   'train_cv_mean_gini': 0.0,
   'train_fit_auc': 0.5,
   'train_fit_gini': 0.0,
   'valid_auc': 0.5,
   'valid_gini': 0.0},
  {'C': 0.0001,
   'train_cv_fold_aucs': [0.5, 0.5, 0.5, 0.5, 0.5],
   'train_cv_fold_ginis': [0.0, 0.0, 0.0, 0.0, 0.0],
   'train_cv_mean_auc': 0.5,
 

# Model Prediction

In [ ]:
df['proba_0'], df['proba_1'], feature_predict_issue = at.logreg_predict(df, best_model, cols_feat_woe)

In [ ]:
df['proba2_0'], df['proba2_1'], feature_predict_issue = at.logreg_predict(df, best_model2, cols_feat_woe)

In [654]:
feature_predict_issue

{'cols_feat_woe_not_in_model': ['NumSatisfactoryTrades_woe',
  'NumTrades60Ever2DerogPubRec_woe',
  'NumTrades90Ever2DerogPubRec_woe',
  'PercentTradesNeverDelq_woe',
  'MSinceMostRecentDelq_woe',
  'MaxDelq2PublicRecLast12M_woe',
  'MaxDelqEver_woe',
  'NumTotalTrades_woe',
  'NumTradesOpeninLast12M_woe',
  'PercentInstallTrades_woe',
  'MSinceMostRecentInqexcl7days_woe',
  'NumInqLast6M_woe',
  'NumInqLast6Mexcl7days_woe',
  'NetFractionRevolvingBurden_woe',
  'NetFractionInstallBurden_woe',
  'NumRevolvingTradesWBalance_woe',
  'NumInstallTradesWBalance_woe',
  'NumBank2NatlTradesWHighUtilization_woe',
  'PercentTradesWBalance_woe'],
 'model_feature_not_in_cols_feat_woe': [],
 'model_feature_missing_from_dataframe': []}

In [ ]:
df[[col_target, 'proba_1']]

,label,proba_1
0,0,0.169436
1,0,0.187412
2,0,0.197447
3,0,0.213071
4,0,0.718495
...,...,...
10454,1,0.594873
10455,0,0.259703
10456,0,0.629000
10457,0,0.791763


In [ ]:
df_stats = at.get_score_predictive_power_data_type(df, 'proba_1', col_type, col_target)

In [657]:
df_stats

,time_period,aucroc,gini,count,count_positive,positive_rate
0,hoot,0.763307,0.526614,2610,1072,0.410728
1,oot,0.806869,0.613737,2576,1335,0.518245
2,test,0.804152,0.608304,1319,649,0.492039
3,train,0.812349,0.624697,2636,1296,0.491654
4,valid,0.793871,0.587742,1318,648,0.491654


In [ ]:
df_stats = at.get_score_predictive_power_timely(df, 'proba_1', col_month, col_target)

In [659]:
df_stats

,time_period,aucroc,gini,count,count_positive,positive_rate
0,201501,0.791027,0.582054,899,432,0.480534
1,201502,0.695316,0.390633,812,322,0.396552
2,201503,0.797062,0.594123,899,318,0.353726
3,201504,0.765661,0.531323,870,418,0.480460
4,201505,0.812279,0.624558,899,525,0.583982
5,201506,0.807871,0.615743,870,426,0.489655
6,201507,0.823421,0.646841,899,427,0.474972
7,201508,0.865093,0.730186,895,472,0.527374
8,201509,0.732702,0.465404,840,325,0.386905
9,201510,0.785931,0.571862,868,435,0.501152


In [ ]:
df_stats = at.get_score_predictive_power_data_type_bootstrap(df, 'proba_1', col_type, col_target)

In [661]:
df_stats

,time_period,CI_5_gini,mean_gini,CI_95_gini,CI_5_aucroc,mean_auc,CI_95_aucroc,mean_positive_rate
0,hoot,0.368493,0.528165,0.681041,0.684246,0.764083,0.840520,0.41057
1,oot,0.463898,0.612629,0.752707,0.731949,0.806315,0.876353,0.51935
2,test,0.455919,0.605379,0.740399,0.727959,0.802690,0.870199,0.49123
3,train,0.473270,0.622213,0.755998,0.736635,0.811107,0.877999,0.49279
4,valid,0.443874,0.589757,0.727280,0.721937,0.794879,0.863640,0.49158


In [ ]:
df_stats = at.compare_score_predictive_power_data_type_bootstrap(df, 'proba_1', 'proba2_1', col_type, col_target)

In [663]:
df_stats

,time_period,champion_CI_5_gini,champion_mean_gini,champion_CI_95_gini,challenger_CI_5_gini,challenger_mean_gini,challenger_CI_95_gini,champion_CI_5_aucroc,champion_mean_auc,champion_CI_95_aucroc,challenger_CI_5_aucroc,challenger_mean_auc,challenger_CI_95_aucroc,mean_positive_rate
0,hoot,0.368493,0.528165,0.681041,0.281612,0.461510,0.635441,0.684246,0.764083,0.840520,0.640806,0.730755,0.817720,0.41057
1,oot,0.465800,0.611789,0.751310,0.328659,0.492202,0.648013,0.732900,0.805894,0.875655,0.664330,0.746101,0.824006,0.51786
2,test,0.459129,0.608827,0.744217,0.336532,0.503516,0.649852,0.729564,0.804413,0.872108,0.668266,0.751758,0.824926,0.49160
3,train,0.489439,0.624309,0.760431,0.359133,0.523884,0.671650,0.744720,0.812154,0.880216,0.679567,0.761942,0.835825,0.49267
4,valid,0.432593,0.587727,0.723612,0.327212,0.493353,0.646689,0.716296,0.793864,0.861806,0.663606,0.746677,0.823345,0.49252


In [ ]:
df_stats = at.get_timely_feature_psi_woe(df.loc[df[col_type]=='train'], df.loc[df[col_type]!='train'], cols_feat_woe, col_month)

In [665]:
df_stats

,time,feature_name,psi,count data
0,201501,ExternalRiskEstimate_woe,0.001409,899
1,201501,MSinceOldestTradeOpen_woe,0.003847,899
2,201501,MSinceMostRecentTradeOpen_woe,0.015159,899
3,201501,AverageMInFile_woe,0.009819,899
4,201501,NumSatisfactoryTrades_woe,0.009620,899
...,...,...,...,...
271,201512,NetFractionInstallBurden_woe,0.019544,868
272,201512,NumRevolvingTradesWBalance_woe,0.019945,868
273,201512,NumInstallTradesWBalance_woe,0.016706,868
274,201512,NumBank2NatlTradesWHighUtilization_woe,0.006037,868


In [ ]:
df_stats = at.get_timely_psi(df.loc[df[col_type]=='train'], df.loc[df[col_type]!='train'], 'proba_1', col_month)

In [667]:
df_stats

,time_period,psi,count data
0,201501,0.014625,899
1,201502,1.065569,812
2,201503,0.225981,899
3,201504,0.062622,416
4,201505,0.076822,459
5,201506,0.046195,433
6,201507,0.062909,455
7,201508,0.008240,438
8,201509,0.107084,436
9,201510,0.024973,868


In [ ]:
df_stats = at.get_timely_target_rate_feature_segment(df.loc[df[col_type]=='train'], cols_feat_woe, col_target, col_month)

In [671]:
df_stats

,time period,feature name,segment,count data,count positive,positive rate
0,201504,ExternalRiskEstimate_woe,-1.352153,95,73,0.768421
1,201504,ExternalRiskEstimate_woe,-0.647546,74,47,0.635135
2,201504,ExternalRiskEstimate_woe,0.019411,64,35,0.546875
3,201504,ExternalRiskEstimate_woe,0.372078,31,11,0.354839
4,201504,ExternalRiskEstimate_woe,0.701260,73,29,0.397260
...,...,...,...,...,...,...
751,201509,PercentTradesWBalance_woe,-0.347168,98,47,0.479592
752,201509,PercentTradesWBalance_woe,0.133993,82,34,0.414634
753,201509,PercentTradesWBalance_woe,0.361089,57,17,0.298246
754,201509,PercentTradesWBalance_woe,0.641411,31,9,0.290323
